In [5]:
!pip install xgboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 114.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 99.6 MB/s eta 0:00:0000:0100:01


In [11]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [31]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from ucimlrepo import fetch_ucirepo

# ---------------------------
# Load Dataset
# ---------------------------
bank_marketing = fetch_ucirepo(id=222)

df = pd.concat([bank_marketing.data.features,
                bank_marketing.data.targets], axis=1)

df.to_csv("bank.csv", index=False)

# ---------------------------
# Encoding Categorical Columns
# ---------------------------
encoders = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# ---------------------------
# Split Data
# ---------------------------
X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# Scaling
# ---------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ---------------------------
# Define Models
# ---------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(eval_metric='logloss')
}

results = {}
trained_models = {}

# ---------------------------
# Train & Evaluate
# ---------------------------
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T
print(results_df)

# ---------------------------
# Save Models + Scaler + Encoders
# ---------------------------
os.makedirs("model", exist_ok=True)

pickle.dump(scaler, open("model/scaler.pkl", "wb"))
pickle.dump(encoders, open("model/encoders.pkl", "wb"))

for name, model in trained_models.items():
    filename = name.replace(" ", "_").lower() + ".pkl"
    pickle.dump(model, open(f"model/{filename}", "wb"))

print("All models, scaler, and encoders saved successfully!")


                     Accuracy       AUC  Precision    Recall        F1  \
Logistic Regression  0.887869  0.870001   0.596977  0.217232  0.318548   
Decision Tree        0.871282  0.695887   0.466421  0.464711  0.465565   
KNN                  0.891187  0.826034   0.585875  0.334555  0.425904   
Naive Bayes          0.824837  0.809394   0.342693  0.492209  0.404063   
Random Forest        0.903351  0.925424   0.651325  0.428048  0.516593   
XGBoost              0.908106  0.928546   0.656250  0.500458  0.567863   

                          MCC  
Logistic Regression  0.313371  
Decision Tree        0.392395  
KNN                  0.388523  
Naive Bayes          0.312110  
Random Forest        0.478118  
XGBoost              0.523443  
All models, scaler, and encoders saved successfully!
